In [ ]:
# ===============================
# SECTION 1: SETUP & DEPENDENCIES
# ===============================
!pip install -q datasets pandas matplotlib seaborn tqdm

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
import numpy as np
import os
import json

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ===============================
# SECTION 2: HC3 DATASET
# ===============================
print("="*60)
print("LOADING HC3 DATASET (Human-ChatGPT Comparison)")
print("="*60)

# Install git-lfs for large files
!git lfs install

# Clone HC3 dataset
!git clone https://huggingface.co/datasets/Hello-SimpleAI/HC3 || echo "Already cloned"


print("\n📁 Checking HC3 directory structure:")
!ls -lh HC3/


hc3_data = {}

# Try loading from the Hugging Face datasets library directly
try:
    print("\n🔄 Method 1: Loading via datasets library...")
    from datasets import load_dataset

    hc3_dataset = load_dataset("Hello-SimpleAI/HC3")

    # Check what splits are available
    print(f"Available splits: {list(hc3_dataset.keys())}")

    # Convert each split to pandas
    for split_name in hc3_dataset.keys():
        hc3_data[split_name] = hc3_dataset[split_name].to_pandas()
        print(f"✓ Loaded {split_name}: {len(hc3_data[split_name])} rows")

except Exception as e:
    print(f"⚠️  Method 1 failed: {e}")
    print("\n🔄 Method 2: Loading from local files...")

    # Try different file formats
    hc3_dir = "HC3"

    for filename in os.listdir(hc3_dir):
        filepath = os.path.join(hc3_dir, filename)

        try:
            if filename.endswith('.parquet'):
                domain = filename.replace('.parquet', '')
                hc3_data[domain] = pd.read_parquet(filepath)
                print(f"✓ Loaded {domain}: {len(hc3_data[domain])} rows")

            elif filename.endswith('.jsonl'):
                domain = filename.replace('.jsonl', '')
                hc3_data[domain] = pd.read_json(filepath, lines=True)
                print(f"✓ Loaded {domain}: {len(hc3_data[domain])} rows")

            elif filename.endswith('.json'):
                domain = filename.replace('.json', '')
                with open(filepath, 'r') as f:
                    data = json.load(f)
                    hc3_data[domain] = pd.DataFrame(data)
                print(f"✓ Loaded {domain}: {len(hc3_data[domain])} rows")

        except Exception as e:
            print(f"✗ Failed to load {filename}: {e}")

# Check if we got any data
if len(hc3_data) == 0:
    print("\n❌ No HC3 data loaded! Let's inspect the structure...")
    !ls -R HC3/ | head -50
    raise ValueError("HC3 dataset loading failed. Check directory structure above.")

print(f"\n✅ Successfully loaded {len(hc3_data)} domain(s)")

# Inspect structure of first dataset
first_domain = list(hc3_data.keys())[0]
print(f"\n🔍 Inspecting structure of '{first_domain}':")
print(f"Columns: {list(hc3_data[first_domain].columns)}")
print(f"\nFirst row sample:")
print(hc3_data[first_domain].head(1))

# ===============================
# SECTION 3: DATA FLATTENING
# ===============================
print("\n" + "="*60)
print("FLATTENING HC3 INTO PAIRED FORMAT")
print("="*60)

def flatten_hc3_robust(df, domain_name):
    """
    Converts HC3 format into paired format
    Handles different possible structures
    """
    rows = []

    for idx, row in df.iterrows():
        question = row.get("question", row.get("prompt", ""))

        # Get human answers (handle different formats)
        if "human_answers" in row:
            human_answers = row["human_answers"]
        elif "human" in row:
            human_answers = row["human"] if isinstance(row["human"], list) else [row["human"]]
        else:
            human_answers = []

        # Get AI answers (handle different formats)
        if "chatgpt_answers" in row:
            ai_answers = row["chatgpt_answers"]
        elif "chatgpt" in row:
            ai_answers = row["chatgpt"] if isinstance(row["chatgpt"], list) else [row["chatgpt"]]
        elif "ai" in row:
            ai_answers = row["ai"] if isinstance(row["ai"], list) else [row["ai"]]
        else:
            ai_answers = []

        # Create pairs
        if len(human_answers) > 0 and len(ai_answers) > 0:
            # Take first human and first AI answer for simplicity
            # (to avoid combinatorial explosion)
            rows.append({
                "question": question,
                "human_answer": human_answers[0],
                "ai_answer": ai_answers[0],
                "domain": domain_name
            })

    return pd.DataFrame(rows)

hc3_flat = {}
for domain, df in hc3_data.items():
    hc3_flat[domain] = flatten_hc3_robust(df, domain)
    print(f"{domain:20} → {len(hc3_flat[domain]):6,} paired examples")

# Combine all domains
if len(hc3_flat) > 0:
    hc3_combined = pd.concat(hc3_flat.values(), ignore_index=True)
    print(f"\n✅ Total paired examples: {len(hc3_combined):,}")
else:
    raise ValueError("No flattened data available!")

# ===============================
# SECTION 4: EDA - BASIC STATISTICS
# ===============================
print("\n" + "="*60)
print("EXPLORATORY DATA ANALYSIS")
print("="*60)

# Check for missing values
print("\n🔍 Data Quality Check:")
print(hc3_combined.isnull().sum())

# Remove rows with missing data
hc3_combined = hc3_combined.dropna()
print(f"After removing nulls: {len(hc3_combined):,} examples")

# Add length metrics
hc3_combined["human_len_words"] = hc3_combined["human_answer"].str.split().str.len()
hc3_combined["ai_len_words"] = hc3_combined["ai_answer"].str.split().str.len()
hc3_combined["question_len_words"] = hc3_combined["question"].str.split().str.len()

hc3_combined["human_len_chars"] = hc3_combined["human_answer"].str.len()
hc3_combined["ai_len_chars"] = hc3_combined["ai_answer"].str.len()

print("\n📊 QUESTION LENGTH STATISTICS (words):")
print(hc3_combined["question_len_words"].describe())

print("\n📊 HUMAN ANSWER LENGTH (words):")
print(hc3_combined["human_len_words"].describe())

print("\n📊 AI ANSWER LENGTH (words):")
print(hc3_combined["ai_len_words"].describe())

# ===============================
# SECTION 5: VISUALIZATION
# ===============================
print("\n" + "="*60)
print("GENERATING VISUALIZATIONS")
print("="*60)

# Plot 1: Length comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(hc3_combined["human_len_words"], bins=50, alpha=0.7, label="Human", color="blue", range=(0, 500))
axes[0].hist(hc3_combined["ai_len_words"], bins=50, alpha=0.7, label="AI", color="red", range=(0, 500))
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Answer Length Distribution: Human vs AI")
axes[0].legend()

# Plot 2: Question length
axes[1].hist(hc3_combined["question_len_words"], bins=50, color="green", alpha=0.7)
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Question Length Distribution")

plt.tight_layout()
plt.show()

# Plot 3: Box plot comparison
plt.figure(figsize=(10, 6))
data_to_plot = [
    hc3_combined["human_len_words"],
    hc3_combined["ai_len_words"]
]
plt.boxplot(data_to_plot, labels=["Human", "AI"])
plt.ylabel("Word Count")
plt.title("Answer Length: Human vs AI (Box Plot)")
plt.grid(axis='y', alpha=0.3)
plt.show()

# ===============================
# SECTION 6: DOMAIN-WISE ANALYSIS
# ===============================
print("\n" + "="*60)
print("DOMAIN-WISE BREAKDOWN")
print("="*60)

domain_stats = hc3_combined.groupby("domain").agg({
    "human_len_words": ["mean", "median", "std"],
    "ai_len_words": ["mean", "median", "std"],
    "question": "count"
}).round(2)

domain_stats.columns = ["Human_Mean", "Human_Median", "Human_Std",
                        "AI_Mean", "AI_Median", "AI_Std", "Count"]
print(domain_stats)

# ===============================
# SECTION 7: SAMPLE INSPECTION
# ===============================
print("\n" + "="*60)
print("SAMPLE EXAMPLES")
print("="*60)

sample_idx = np.random.randint(0, len(hc3_combined))
sample = hc3_combined.iloc[sample_idx]

print(f"\n🔍 Random Sample #{sample_idx}")
print(f"Domain: {sample['domain']}")
print(f"\nQuestion ({sample['question_len_words']} words):")
print(sample['question'])
print(f"\nHuman Answer ({sample['human_len_words']} words):")
print(sample['human_answer'][:300] + "..." if len(sample['human_answer']) > 300 else sample['human_answer'])
print(f"\nAI Answer ({sample['ai_len_words']} words):")
print(sample['ai_answer'][:300] + "..." if len(sample['ai_answer']) > 300 else sample['ai_answer'])

# ===============================
# SECTION 8: SAVE PROCESSED DATA
# ===============================
print("\n" + "="*60)
print("SAVING PROCESSED DATASET")
print("="*60)

# Save for later stages
hc3_combined.to_csv("hc3_paired_dataset.csv", index=False)
print("✓ Saved to: hc3_paired_dataset.csv")
print(f"✓ Total examples: {len(hc3_combined):,}")

print("\n🎯 Dataset ready for Stage 1: Feature Extraction!")

In [ ]:
# ===============================
# EDA + STANDARDIZATION SCRIPT
# ===============================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
import numpy as np
import json

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# ===============================
# PART 1: ELI5 DATASET - LOAD & EDA
# ===============================
print("="*70)
print("PART 1: ELI5 DATASET (Human-Only Answers)")
print("="*70)

# Load ELI5
eli5 = load_dataset("sentence-transformers/eli5")
eli5_df = eli5["train"].to_pandas()

print(f"\n📊 ELI5 Dataset Shape: {eli5_df.shape}")
print(f"Columns: {list(eli5_df.columns)}")

# Check for missing values
print("\n🔍 Missing Values:")
print(eli5_df.isnull().sum())

# Add length metrics
eli5_df["question_len_words"] = eli5_df["question"].str.split().str.len()
eli5_df["answer_len_words"] = eli5_df["answer"].str.split().str.len()
eli5_df["question_len_chars"] = eli5_df["question"].str.len()
eli5_df["answer_len_chars"] = eli5_df["answer"].str.len()

print("\n📊 ELI5 QUESTION LENGTH (words):")
print(eli5_df["question_len_words"].describe())

print("\n📊 ELI5 ANSWER LENGTH (words):")
print(eli5_df["answer_len_words"].describe())

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(eli5_df["question_len_words"], bins=50, color="purple", alpha=0.7)
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Frequency")
axes[0].set_title("ELI5: Question Length Distribution")
axes[0].set_xlim(0, 100)

axes[1].hist(eli5_df["answer_len_words"], bins=50, color="orange", alpha=0.7)
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")
axes[1].set_title("ELI5: Answer Length Distribution")
axes[1].set_xlim(0, 500)

plt.tight_layout()
plt.show()

# Sample inspection
print("\n" + "="*70)
print("ELI5 SAMPLE EXAMPLES")
print("="*70)

for i in range(3):
    sample = eli5_df.sample(1).iloc[0]
    print(f"\n🔍 Sample {i+1}")
    print(f"Question ({sample['question_len_words']} words): {sample['question']}")
    print(f"Answer ({sample['answer_len_words']} words): {sample['answer'][:250]}...")
    print("-"*70)

# ===============================
# PART 2: HC3 DATASET - ALREADY LOADED
# ===============================
print("\n" + "="*70)
print("PART 2: HC3 DATASET (Human + ChatGPT Answers)")
print("="*70)

# Load the processed HC3 (from previous script)
hc3_combined = pd.read_csv("hc3_paired_dataset.csv")

print(f"\n📊 HC3 Dataset Shape: {hc3_combined.shape}")
print(f"Columns: {list(hc3_combined.columns)}")

print("\n🔍 Missing Values:")
print(hc3_combined.isnull().sum())

print("\n📊 HC3 Domain Distribution:")
print(hc3_combined["domain"].value_counts())

# ===============================
# PART 3: CSV STRUCTURE VISUALIZATION
# ===============================
print("\n" + "="*70)
print("PART 3: CSV STRUCTURE PREVIEW")
print("="*70)

print("\n📄 HC3 DATASET STRUCTURE (First 3 rows):")
print(hc3_combined.head(3).to_string())

print("\n\n📄 ELI5 DATASET STRUCTURE (First 3 rows):")
print(eli5_df.head(3).to_string())

# ===============================
# PART 4: STANDARDIZE COLUMN NAMES
# ===============================
print("\n" + "="*70)
print("PART 4: STANDARDIZING COLUMN NAMES")
print("="*70)

# HC3: Rename to standard format
hc3_standardized = hc3_combined.rename(columns={
    "human_answer": "human_answer",  # already correct
    "ai_answer": "llm_answer",        # CHANGE: ai_answer → llm_answer
    "question": "question",           # already correct
    "domain": "domain"                # already correct
})

# Select only core columns
hc3_standardized = hc3_standardized[[
    "question",
    "human_answer",
    "llm_answer",
    "domain"
]]

print("✅ HC3 Standardized Columns:", list(hc3_standardized.columns))

# ELI5: Rename to standard format
eli5_standardized = eli5_df.rename(columns={
    "question": "question",
    "answer": "human_answer"  # CHANGE: answer → human_answer
})

# Add domain and placeholder for llm_answer
eli5_standardized["domain"] = "eli5"
eli5_standardized["llm_answer"] = None  # No LLM answers yet

# Select only core columns
eli5_standardized = eli5_standardized[[
    "question",
    "human_answer",
    "llm_answer",
    "domain"
]]

print("✅ ELI5 Standardized Columns:", list(eli5_standardized.columns))

# ===============================
# PART 5: SAVE AS CSV
# ===============================
print("\n" + "="*70)
print("PART 5: SAVING STANDARDIZED CSV FILES")
print("="*70)

hc3_standardized.to_csv("hc3_standardized.csv", index=False)
print(f"✅ Saved: hc3_standardized.csv ({len(hc3_standardized):,} rows)")

eli5_standardized.to_csv("eli5_standardized.csv", index=False)
print(f"✅ Saved: eli5_standardized.csv ({len(eli5_standardized):,} rows)")

# Preview standardized CSVs
print("\n📄 HC3 STANDARDIZED PREVIEW:")
print(hc3_standardized.head(2))

print("\n📄 ELI5 STANDARDIZED PREVIEW:")
print(eli5_standardized.head(2))

# ===============================
# PART 6: SAVE AS JSON FILES
# ===============================
print("\n" + "="*70)
print("PART 6: SAVING AS JSON FILES")
print("="*70)

# HC3 → JSON
hc3_json = hc3_standardized.to_dict(orient="records")
with open("hc3_standardized.json", "w") as f:
    json.dump(hc3_json, f, indent=2)
print(f"✅ Saved: hc3_standardized.json ({len(hc3_json):,} records)")

# ELI5 → JSON
eli5_json = eli5_standardized.to_dict(orient="records")
with open("eli5_standardized.json", "w") as f:
    json.dump(eli5_json, f, indent=2)
print(f"✅ Saved: eli5_standardized.json ({len(eli5_json):,} records)")

# Show JSON structure example
print("\n📄 HC3 JSON STRUCTURE (First record):")
print(json.dumps(hc3_json[0], indent=2))

print("\n📄 ELI5 JSON STRUCTURE (First record):")
print(json.dumps(eli5_json[0], indent=2))

# ===============================
# PART 7: COMPARATIVE EDA
# ===============================
print("\n" + "="*70)
print("PART 7: COMPARATIVE ANALYSIS (HC3 vs ELI5)")
print("="*70)

# Calculate lengths for HC3
hc3_standardized["question_len"] = hc3_standardized["question"].str.split().str.len()
hc3_standardized["human_len"] = hc3_standardized["human_answer"].str.split().str.len()
hc3_standardized["llm_len"] = hc3_standardized["llm_answer"].str.split().str.len()

# Calculate lengths for ELI5
eli5_standardized["question_len"] = eli5_standardized["question"].str.split().str.len()
eli5_standardized["human_len"] = eli5_standardized["human_answer"].str.split().str.len()

# Comparison table
comparison = pd.DataFrame({
    "Dataset": ["HC3", "ELI5"],
    "Total_Samples": [len(hc3_standardized), len(eli5_standardized)],
    "Avg_Question_Len": [
        hc3_standardized["question_len"].mean(),
        eli5_standardized["question_len"].mean()
    ],
    "Avg_Human_Answer_Len": [
        hc3_standardized["human_len"].mean(),
        eli5_standardized["human_len"].mean()
    ],
    "Has_LLM_Answers": ["Yes", "No"]
}).round(2)

print("\n📊 DATASET COMPARISON:")
print(comparison.to_string(index=False))

# Side-by-side visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# HC3 Human vs LLM
axes[0, 0].hist(hc3_standardized["human_len"], bins=50, alpha=0.7, label="Human", color="blue", range=(0, 400))
axes[0, 0].hist(hc3_standardized["llm_len"], bins=50, alpha=0.7, label="LLM", color="red", range=(0, 400))
axes[0, 0].set_title("HC3: Answer Length Distribution")
axes[0, 0].set_xlabel("Word Count")
axes[0, 0].legend()

# HC3 Questions
axes[0, 1].hist(hc3_standardized["question_len"], bins=50, color="green", alpha=0.7)
axes[0, 1].set_title("HC3: Question Length")
axes[0, 1].set_xlabel("Word Count")

# ELI5 Answers
axes[1, 0].hist(eli5_standardized["human_len"], bins=50, color="purple", alpha=0.7, range=(0, 400))
axes[1, 0].set_title("ELI5: Answer Length Distribution")
axes[1, 0].set_xlabel("Word Count")

# ELI5 Questions
axes[1, 1].hist(eli5_standardized["question_len"], bins=50, color="orange", alpha=0.7)
axes[1, 1].set_title("ELI5: Question Length")
axes[1, 1].set_xlabel("Word Count")

plt.tight_layout()
plt.show()

# ===============================
# PART 8: DOMAIN BREAKDOWN (HC3 ONLY)
# ===============================
print("\n" + "="*70)
print("PART 8: HC3 DOMAIN-WISE STATISTICS")
print("="*70)

domain_breakdown = hc3_standardized.groupby("domain").agg({
    "question": "count",
    "question_len": "mean",
    "human_len": "mean",
    "llm_len": "mean"
}).round(2)

domain_breakdown.columns = ["Sample_Count", "Avg_Q_Len", "Avg_Human_Len", "Avg_LLM_Len"]
print(domain_breakdown)

# ===============================
# SUMMARY
# ===============================
print("\n" + "="*70)
print("🎯 SUMMARY - FILES CREATED")
print("="*70)
print("✅ hc3_standardized.csv  - HC3 with standardized columns")
print("✅ hc3_standardized.json - HC3 in JSON format")
print("✅ eli5_standardized.csv - ELI5 with standardized columns")
print("✅ eli5_standardized.json - ELI5 in JSON format")

In [ ]:
# ===============================
# REMOVE DUPLICATES FROM HC3
# ===============================

# Load the standardized HC3
hc3_standardized = pd.read_csv("hc3_standardized.csv")

print(f"Original HC3 size: {len(hc3_standardized):,} records")
print(f"\nDomain distribution:")
print(hc3_standardized["domain"].value_counts())

# Option 1: Remove the "all" domain (it's a duplicate merge)
hc3_dedup = hc3_standardized[hc3_standardized["domain"] != "all"].copy()
print(f"\n✅ After removing 'all' domain: {len(hc3_dedup):,} records")

# Option 2: Also remove any exact duplicate rows (based on question + answers)
hc3_dedup = hc3_dedup.drop_duplicates(
    subset=["question", "human_answer", "llm_answer"],
    keep="first"
)
print(f"✅ After removing exact duplicates: {len(hc3_dedup):,} records")

print(f"\n📊 Final domain distribution:")
print(hc3_dedup["domain"].value_counts())

# Re-save cleaned files
hc3_dedup.to_csv("hc3_standardized.csv", index=False)
print(f"\n✅ Saved: hc3_standardized.csv ({len(hc3_dedup):,} rows)")

# Re-save JSON
hc3_json = hc3_dedup.to_dict(orient="records")
with open("hc3_standardized.json", "w") as f:
    json.dump(hc3_json, f, indent=2)
print(f"✅ Saved: hc3_standardized.json ({len(hc3_json):,} records)")


In [ ]:
# ===============================
# MISTRAL 7B - LLM ANSWER GENERATION
# ===============================

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm
import json
import gc

# ===============================
#  CONFIGURATION
# ===============================
CONFIG = {
    # Data Settings
    "SAMPLE_SIZE": 15000,           # Number of samples to process
    "RANDOM_STATE": 42,              # For reproducibility
    "INPUT_FILE": "eli5_standardized.csv",
    "OUTPUT_CSV": "dataset_50k_with_llm.csv",
    "OUTPUT_JSON": "dataset_50k_with_llm.json",

    # Model Settings
    "MODEL_NAME": "mistralai/Mistral-7B-Instruct-v0.2",
    "USE_FLASH_ATTENTION": True,     # Faster on A100
    "USE_TORCH_COMPILE": True,       # ~30% speedup

    # Generation Settings
    "BATCH_SIZE": 48,                # 32-96 for A100 (adjust based on VRAM)
    "MAX_NEW_TOKENS": 150,           # Max answer length
    "MAX_INPUT_LENGTH": 512,         # Max question length
    "TEMPERATURE": 0.7,              # Randomness (0.7-0.9 recommended)
    "TOP_P": 0.9,                    # Nucleus sampling
    "DO_SAMPLE": True,               # Enable sampling

    # Quality Check
    "NUM_SAMPLES_TO_DISPLAY": 3      # Samples to show at end
}

print("="*70)
print("⚙️  CONFIGURATION")
print("="*70)
for key, value in CONFIG.items():
    print(f"{key:25s}: {value}")
print("="*70)

# ===============================
# STEP 1: LOAD & SAMPLE DATASET
# ===============================
print("\n" + "="*70)
print(f"STEP 1: LOADING {CONFIG['SAMPLE_SIZE']:,} SAMPLES FROM ELI5")
print("="*70)

eli5 = pd.read_csv(CONFIG["INPUT_FILE"])
print(f"ELI5 size: {len(eli5):,}")

# Sample
if len(eli5) > CONFIG["SAMPLE_SIZE"]:
    sampled_data = eli5.sample(
        n=CONFIG["SAMPLE_SIZE"],
        random_state=CONFIG["RANDOM_STATE"]
    ).reset_index(drop=True)
else:
    sampled_data = eli5.copy()

print(f"✅ Sampled: {len(sampled_data):,} examples")
print(f"Domain: {sampled_data['domain'].unique()}")

# ===============================
# STEP 2: LOAD MISTRAL 7B MODEL
# ===============================
print("\n" + "="*70)
print("STEP 2: LOADING MISTRAL 7B MODEL")
print("="*70)

# Install flash attention if needed
if CONFIG["USE_FLASH_ATTENTION"]:
    !pip install -q flash-attn --no-build-isolation

# # Quantization config
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True
# )

# print(f"Loading {CONFIG['MODEL_NAME']}...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL_NAME"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# ===============================
# Load model - PURE FP16 (No 4-bit)
# ===============================
model_kwargs = {
    "device_map": "auto",
    "torch_dtype": torch.float16,  # Pure fp16
}

if CONFIG["USE_FLASH_ATTENTION"]:
    model_kwargs["attn_implementation"] = "flash_attention_2"
    print("✅ Flash Attention 2 enabled")

print("🚀 Loading model in pure FP16 (no quantization)...")

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["MODEL_NAME"],
    **model_kwargs
)

# Compile model for speedup
if CONFIG["USE_TORCH_COMPILE"]:
    print("🚀 Compiling model with torch.compile()...")
    model = torch.compile(model, mode="reduce-overhead")

print(f"✅ Model loaded on: {model.device}")

# ===============================
# STEP 3: DEFINE PROMPT TEMPLATE
# ===============================

def create_prompt(question):
    """Simple, neutral prompt"""
    prompt = f"""<s>[INST] Answer the following question clearly and concisely.

Question: {question}

Answer: [/INST]"""
    return prompt

# ===============================
# STEP 4: BATCH GENERATION
# ===============================

def generate_batch_answers(questions, batch_size, max_new_tokens):
    """Generate answers in batches"""
    answers = []

    # Change this line:
    for i in tqdm(range(0, len(questions), batch_size),
                  desc="Generating",
                  position=0,
                  leave=True):
        batch_questions = questions[i:i+batch_size]
        prompts = [create_prompt(q) for q in batch_questions]

        # Tokenize
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=CONFIG["MAX_INPUT_LENGTH"]
        ).to(model.device)

        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=CONFIG["TEMPERATURE"],
                top_p=CONFIG["TOP_P"],
                do_sample=CONFIG["DO_SAMPLE"],
                pad_token_id=tokenizer.eos_token_id
            )


        for j, output in enumerate(outputs):
            prompt_length = inputs['input_ids'][j].shape[0]
            generated_tokens = output[prompt_length:]
            answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            answers.append(answer.strip())

    return answers

# ===============================
# STEP 5: GENERATE LLM ANSWERS
# ===============================
print("\n" + "="*70)
print("STEP 5: GENERATING LLM ANSWERS")
print("="*70)

needs_generation = sampled_data["llm_answer"].isna()
questions_to_generate = sampled_data[needs_generation]["question"].tolist()

print(f"Questions needing answers: {len(questions_to_generate):,}")
print(f"Batch size: {CONFIG['BATCH_SIZE']}")
print(f"Max tokens per answer: {CONFIG['MAX_NEW_TOKENS']}")

if len(questions_to_generate) > 0:
    print(f"\n🚀 Starting generation...")

    generated_answers = generate_batch_answers(
        questions_to_generate,
        batch_size=CONFIG["BATCH_SIZE"],
        max_new_tokens=CONFIG["MAX_NEW_TOKENS"]
    )

    sampled_data.loc[needs_generation, "llm_answer"] = generated_answers
    print(f"\n✅ Generated {len(generated_answers):,} answers")
else:
    print("All questions already have LLM answers!")

# ===============================
# STEP 6: SAVE RESULTS
# ===============================
print("\n" + "="*70)
print("STEP 6: SAVING RESULTS")
print("="*70)

missing = sampled_data["llm_answer"].isna().sum()
print(f"Missing answers: {missing}")

# Save CSV
sampled_data.to_csv(CONFIG["OUTPUT_CSV"], index=False)
print(f"✅ Saved: {CONFIG['OUTPUT_CSV']} ({len(sampled_data):,} rows)")

# Save JSON
data_json = sampled_data.to_dict(orient="records")
with open(CONFIG["OUTPUT_JSON"], "w") as f:
    json.dump(data_json, f, indent=2)
print(f"✅ Saved: {CONFIG['OUTPUT_JSON']}")

# ===============================
# STEP 7: QUALITY CHECK
# ===============================
print("\n" + "="*70)
print("STEP 7: QUALITY CHECK")
print("="*70)

for i in range(CONFIG["NUM_SAMPLES_TO_DISPLAY"]):
    sample = sampled_data.sample(1).iloc[0]
    print(f"\n{'='*70}")
    print(f"Sample {i+1} - Domain: {sample['domain']}")
    print(f"\nQ: {sample['question']}")
    print(f"\nHuman: {sample['human_answer'][:200]}...")
    print(f"\nMistral: {sample['llm_answer'][:200]}...")

print("\n" + "="*70)
print("🎯 GENERATION COMPLETE!")
print("="*70)
print(f"Total samples: {len(sampled_data):,}")
print(f"Config used: batch_size={CONFIG['BATCH_SIZE']}, max_tokens={CONFIG['MAX_NEW_TOKENS']}")

In [ ]:
# ===============================
# SANITY CHECK & EDA ON GENERATED DATA
# ===============================

import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter
import re

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# ===============================
# STEP 1: LOAD GENERATED DATA
# ===============================
print("="*70)
print("STEP 1: LOADING GENERATED DATA")
print("="*70)

# Load the JSON file
with open("/content/dataset_50k_with_llm.json", "r") as f:
    data_json = json.load(f)

df = pd.DataFrame(data_json)

print(f"✅ Loaded: {len(df):,} samples")
print(f"Columns: {list(df.columns)}")

# ===============================
# STEP 2: DATA COMPLETENESS CHECK
# ===============================
print("\n" + "="*70)
print("STEP 2: DATA COMPLETENESS CHECK")
print("="*70)

print("\n🔍 Missing Values:")
missing = df.isnull().sum()
print(missing)

if missing.sum() > 0:
    print("\n⚠️  WARNING: Found missing values!")
    print(df[df.isnull().any(axis=1)].head())
else:
    print("\n✅ No missing values - data is complete!")

# Check for empty strings
print("\n🔍 Empty String Check:")
empty_questions = (df['question'].str.strip() == '').sum()
empty_human = (df['human_answer'].str.strip() == '').sum()
empty_llm = (df['llm_answer'].str.strip() == '').sum()

print(f"Empty questions: {empty_questions}")
print(f"Empty human answers: {empty_human}")
print(f"Empty LLM answers: {empty_llm}")

# ===============================
# STEP 3: LENGTH ANALYSIS
# ===============================
print("\n" + "="*70)
print("STEP 3: LENGTH ANALYSIS")
print("="*70)

# Calculate lengths
df['question_len_words'] = df['question'].str.split().str.len()
df['question_len_chars'] = df['question'].str.len()

df['human_len_words'] = df['human_answer'].str.split().str.len()
df['human_len_chars'] = df['human_answer'].str.len()

df['llm_len_words'] = df['llm_answer'].str.split().str.len()
df['llm_len_chars'] = df['llm_answer'].str.len()

print("\n📊 QUESTION LENGTH STATISTICS:")
print(df['question_len_words'].describe())

print("\n📊 HUMAN ANSWER LENGTH STATISTICS:")
print(df['human_len_words'].describe())

print("\n📊 LLM ANSWER LENGTH STATISTICS:")
print(df['llm_len_words'].describe())

# Comparison
print("\n📊 HUMAN vs LLM COMPARISON:")
comparison = pd.DataFrame({
    'Metric': ['Mean', 'Median', 'Std', 'Min', 'Max'],
    'Human (words)': [
        df['human_len_words'].mean(),
        df['human_len_words'].median(),
        df['human_len_words'].std(),
        df['human_len_words'].min(),
        df['human_len_words'].max()
    ],
    'LLM (words)': [
        df['llm_len_words'].mean(),
        df['llm_len_words'].median(),
        df['llm_len_words'].std(),
        df['llm_len_words'].min(),
        df['llm_len_words'].max()
    ]
})
print(comparison.round(2))

# ===============================
# STEP 4: VISUALIZATIONS
# ===============================
print("\n" + "="*70)
print("STEP 4: GENERATING VISUALIZATIONS")
print("="*70)

# Plot 1: Length distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Question length
axes[0, 0].hist(df['question_len_words'], bins=50, color='green', alpha=0.7)
axes[0, 0].set_title('Question Length Distribution')
axes[0, 0].set_xlabel('Words')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['question_len_words'].mean(), color='red', linestyle='--', label='Mean')
axes[0, 0].legend()

# Human vs LLM answer length
axes[0, 1].hist(df['human_len_words'], bins=50, alpha=0.7, label='Human', color='blue', range=(0, 500))
axes[0, 1].hist(df['llm_len_words'], bins=50, alpha=0.7, label='LLM', color='red', range=(0, 500))
axes[0, 1].set_title('Answer Length: Human vs LLM')
axes[0, 1].set_xlabel('Words')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()

# Box plot comparison
box_data = [df['human_len_words'], df['llm_len_words']]
axes[1, 0].boxplot(box_data, labels=['Human', 'LLM'])
axes[1, 0].set_title('Answer Length Distribution (Box Plot)')
axes[1, 0].set_ylabel('Words')
axes[1, 0].grid(axis='y', alpha=0.3)

# Scatter plot: Human vs LLM length
axes[1, 1].scatter(df['human_len_words'], df['llm_len_words'], alpha=0.3, s=10)
axes[1, 1].set_xlabel('Human Answer Length (words)')
axes[1, 1].set_ylabel('LLM Answer Length (words)')
axes[1, 1].set_title('Human vs LLM Answer Length Correlation')
axes[1, 1].plot([0, 500], [0, 500], 'r--', alpha=0.5, label='y=x')
axes[1, 1].legend()
axes[1, 1].set_xlim(0, 500)
axes[1, 1].set_ylim(0, 500)

plt.tight_layout()
plt.show()

# ===============================
# STEP 5: TEXT QUALITY CHECKS
# ===============================
print("\n" + "="*70)
print("STEP 5: TEXT QUALITY CHECKS")
print("="*70)

# Check for suspiciously short LLM answers
short_llm = df[df['llm_len_words'] < 10]
print(f"\n⚠️  LLM answers with < 10 words: {len(short_llm)}")

if len(short_llm) > 0:
    print("\nSample short LLM answers:")
    for i, row in short_llm.head(3).iterrows():
        print(f"\nQuestion: {row['question']}")
        print(f"LLM Answer ({row['llm_len_words']} words): {row['llm_answer']}")

# Check for suspiciously long LLM answers (might be truncated or repeated)
long_llm = df[df['llm_len_words'] > 400]
print(f"\n📏 LLM answers with > 400 words: {len(long_llm)}")

# Check for duplicate LLM answers (potential generation issue)
duplicate_llm = df['llm_answer'].duplicated().sum()
print(f"\n🔁 Duplicate LLM answers: {duplicate_llm}")

if duplicate_llm > 0:
    print("\nMost common duplicate:")
    duplicates = df[df['llm_answer'].duplicated(keep=False)]
    print(duplicates['llm_answer'].value_counts().head(3))

# ===============================
# STEP 6: SAMPLE INSPECTION
# ===============================
print("\n" + "="*70)
print("STEP 6: SAMPLE INSPECTION (Random Examples)")
print("="*70)

for i in range(5):
    sample = df.sample(1).iloc[0]
    print(f"\n{'='*70}")
    print(f"SAMPLE {i+1}")
    print(f"{'='*70}")
    print(f"\n📝 Question ({sample['question_len_words']} words):")
    print(f"{sample['question']}")
    print(f"\n👤 Human Answer ({sample['human_len_words']} words):")
    print(f"{sample['human_answer'][:300]}{'...' if len(sample['human_answer']) > 300 else ''}")
    print(f"\n🤖 LLM Answer - Mistral ({sample['llm_len_words']} words):")
    print(f"{sample['llm_answer'][:300]}{'...' if len(sample['llm_answer']) > 300 else ''}")

# ===============================
# STEP 7: STYLISTIC ANALYSIS
# ===============================
print("\n" + "="*70)
print("STEP 7: STYLISTIC ANALYSIS")
print("="*70)

def calculate_lexical_diversity(text):
    """Calculate type-token ratio (unique words / total words)"""
    words = text.lower().split()
    if len(words) == 0:
        return 0
    return len(set(words)) / len(words)

def count_sentences(text):
    """Rough sentence count"""
    return len(re.findall(r'[.!?]+', text))

# Calculate metrics
df['human_lexical_diversity'] = df['human_answer'].apply(calculate_lexical_diversity)
df['llm_lexical_diversity'] = df['llm_answer'].apply(calculate_lexical_diversity)

df['human_sentence_count'] = df['human_answer'].apply(count_sentences)
df['llm_sentence_count'] = df['llm_answer'].apply(count_sentences)

print("\n📊 LEXICAL DIVERSITY (higher = more varied vocabulary):")
print(f"Human: {df['human_lexical_diversity'].mean():.3f} ± {df['human_lexical_diversity'].std():.3f}")
print(f"LLM:   {df['llm_lexical_diversity'].mean():.3f} ± {df['llm_lexical_diversity'].std():.3f}")

print("\n📊 AVERAGE SENTENCE COUNT:")
print(f"Human: {df['human_sentence_count'].mean():.1f} sentences")
print(f"LLM:   {df['llm_sentence_count'].mean():.1f} sentences")

# ===============================
# STEP 8: SUMMARY STATISTICS
# ===============================
print("\n" + "="*70)
print("STEP 8: SUMMARY REPORT")
print("="*70)

print(f"\n✅ DATASET OVERVIEW:")
print(f"   Total samples: {len(df):,}")
print(f"   Domain: {df['domain'].unique()[0]}")
print(f"   Completeness: {100 - (missing.sum() / len(df) * 100):.1f}%")

print(f"\n📊 ANSWER LENGTH:")
print(f"   Human: {df['human_len_words'].mean():.1f} ± {df['human_len_words'].std():.1f} words")
print(f"   LLM:   {df['llm_len_words'].mean():.1f} ± {df['llm_len_words'].std():.1f} words")
print(f"   Ratio (LLM/Human): {(df['llm_len_words'].mean() / df['human_len_words'].mean()):.2f}x")

print(f"\n🎯 QUALITY FLAGS:")
print(f"   Short LLM answers (<10 words): {len(short_llm)}")
print(f"   Duplicate LLM answers: {duplicate_llm}")
print(f"   Empty responses: {empty_llm}")

# Save augmented dataframe with metrics
df.to_csv("eli5_with_metrics.csv", index=False)
print(f"\n💾 Saved augmented data with metrics: eli5_with_metrics.csv")

print("\n" + "="*70)
print("🎯 SANITY CHECK COMPLETE!")
print("="*70)
print("\n✅ Data looks good - ready to proceed to Stage 1: Feature Extraction")

In [ ]:
# ===============================
# STAGE 1: DATA PREPARATION FOR DETECTOR TRAINING
# ===============================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# ===============================
# STEP 1: LOAD BOTH DATASETS
# ===============================
print("="*70)
print("STAGE 1: DATA PREPARATION")
print("="*70)

# Load JSON files instead
import json

# Load HC3
with open("/content/hc3_standardized.json", "r") as f:
    hc3_data = json.load(f)
hc3 = pd.DataFrame(hc3_data)
print(f"HC3 loaded: {len(hc3):,} samples")

# Load ELI5
with open("/content/dataset_50k_with_llm.json", "r") as f:
    eli5_data = json.load(f)
eli5 = pd.DataFrame(eli5_data)
print(f"ELI5 loaded: {len(eli5):,} samples")
# ===============================
# STEP 2: CREATE BINARY CLASSIFICATION FORMAT
# ===============================
print("\n" + "="*70)
print("STEP 2: CREATING BINARY CLASSIFICATION DATASETS")
print("="*70)

def create_binary_dataset(df, dataset_name):
    """
    Convert paired format to binary classification format
    Input: answer text only
    Output: label (human/llm)
    """
    rows = []

    # Human answers
    for _, row in df.iterrows():
        rows.append({
            'text': row['human_answer'],
            'label': 'human',
            'domain': row['domain'],
            'dataset': dataset_name
        })

    # LLM answers
    for _, row in df.iterrows():
        rows.append({
            'text': row['llm_answer'],
            'label': 'llm',
            'domain': row['domain'],
            'dataset': dataset_name
        })

    return pd.DataFrame(rows)

# Create binary datasets
hc3_binary = create_binary_dataset(hc3, 'hc3')
eli5_binary = create_binary_dataset(eli5, 'eli5')

print(f"HC3 binary: {len(hc3_binary):,} samples")
print(f"  - Human: {(hc3_binary['label'] == 'human').sum():,}")
print(f"  - LLM:   {(hc3_binary['label'] == 'llm').sum():,}")

print(f"\nELI5 binary: {len(eli5_binary):,} samples")
print(f"  - Human: {(eli5_binary['label'] == 'human').sum():,}")
print(f"  - LLM:   {(eli5_binary['label'] == 'llm').sum():,}")

# ===============================
# STEP 3: LENGTH MATCHING (CRITICAL)
# ===============================
print("\n" + "="*70)
print("STEP 3: LENGTH MATCHING")
print("="*70)

def length_match_dataset(df, tolerance=0.2):
    """
    Create length-matched subset to avoid length-based detection artifacts
    tolerance: acceptable length difference (20% by default)
    """
    df = df.copy()
    df['text_len'] = df['text'].str.split().str.len()

    human_df = df[df['label'] == 'human'].copy()
    llm_df = df[df['label'] == 'llm'].copy()

    # Match each human answer with closest LLM answer by length
    matched_pairs = []

    for idx, human_row in human_df.iterrows():
        human_len = human_row['text_len']

        # Find LLM answers within tolerance
        llm_subset = llm_df[
            (llm_df['text_len'] >= human_len * (1 - tolerance)) &
            (llm_df['text_len'] <= human_len * (1 + tolerance))
        ]

        if len(llm_subset) > 0:
            # Pick random LLM answer from valid range
            llm_match = llm_subset.sample(1).iloc[0]
            matched_pairs.append(human_row)
            matched_pairs.append(llm_match)

    matched_df = pd.DataFrame(matched_pairs)
    return matched_df.reset_index(drop=True)

# Create length-matched versions
hc3_matched = length_match_dataset(hc3_binary)
eli5_matched = length_match_dataset(eli5_binary)

print(f"HC3 length-matched: {len(hc3_matched):,} samples")
print(f"ELI5 length-matched: {len(eli5_matched):,} samples")

# Verify matching worked
hc3_matched['text_len'] = hc3_matched['text'].str.split().str.len()
print(f"\nHC3 Length Stats (matched):")
print(hc3_matched.groupby('label')['text_len'].describe())

eli5_matched['text_len'] = eli5_matched['text'].str.split().str.len()
print(f"\nELI5 Length Stats (matched):")
print(eli5_matched.groupby('label')['text_len'].describe())

# ===============================
# STEP 4: TRAIN/TEST SPLITS
# ===============================
print("\n" + "="*70)
print("STEP 4: CREATING TRAIN/TEST SPLITS")
print("="*70)

# For each dataset: 80/20 split
hc3_train, hc3_test = train_test_split(
    hc3_matched,
    test_size=0.2,
    random_state=42,
    stratify=hc3_matched['label']
)

eli5_train, eli5_test = train_test_split(
    eli5_matched,
    test_size=0.2,
    random_state=42,
    stratify=eli5_matched['label']
)

print(f"HC3 splits:")
print(f"  Train: {len(hc3_train):,} | Test: {len(hc3_test):,}")
print(f"  Train balance: {hc3_train['label'].value_counts().to_dict()}")

print(f"\nELI5 splits:")
print(f"  Train: {len(eli5_train):,} | Test: {len(eli5_test):,}")
print(f"  Train balance: {eli5_train['label'].value_counts().to_dict()}")

# ===============================
# STEP 5: SAVE PREPARED DATASETS
# ===============================
print("\n" + "="*70)
print("STEP 5: SAVING PREPARED DATASETS")
print("="*70)

# Save all splits
hc3_train.to_csv("hc3_train.csv", index=False)
hc3_test.to_csv("hc3_test.csv", index=False)
eli5_train.to_csv("eli5_train.csv", index=False)
eli5_test.to_csv("eli5_test.csv", index=False)

print("✅ Saved train/test splits:")
print("   - hc3_train.csv")
print("   - hc3_test.csv")
print("   - eli5_train.csv")
print("   - eli5_test.csv")

# ===============================
# STEP 6: VISUALIZATION - LENGTH DISTRIBUTIONS
# ===============================
print("\n" + "="*70)
print("STEP 6: LENGTH DISTRIBUTION VISUALIZATION")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# HC3 - Before matching
hc3_binary['text_len'] = hc3_binary['text'].str.split().str.len()
axes[0, 0].hist(hc3_binary[hc3_binary['label']=='human']['text_len'],
                bins=50, alpha=0.7, label='Human', range=(0, 500))
axes[0, 0].hist(hc3_binary[hc3_binary['label']=='llm']['text_len'],
                bins=50, alpha=0.7, label='LLM', range=(0, 500))
axes[0, 0].set_title('HC3 - Before Length Matching')
axes[0, 0].set_xlabel('Length (words)')
axes[0, 0].legend()

# HC3 - After matching
axes[0, 1].hist(hc3_matched[hc3_matched['label']=='human']['text_len'],
                bins=50, alpha=0.7, label='Human', range=(0, 500))
axes[0, 1].hist(hc3_matched[hc3_matched['label']=='llm']['text_len'],
                bins=50, alpha=0.7, label='LLM', range=(0, 500))
axes[0, 1].set_title('HC3 - After Length Matching')
axes[0, 1].set_xlabel('Length (words)')
axes[0, 1].legend()

# ELI5 - Before matching
eli5_binary['text_len'] = eli5_binary['text'].str.split().str.len()
axes[1, 0].hist(eli5_binary[eli5_binary['label']=='human']['text_len'],
                bins=50, alpha=0.7, label='Human', range=(0, 500))
axes[1, 0].hist(eli5_binary[eli5_binary['label']=='llm']['text_len'],
                bins=50, alpha=0.7, label='LLM', range=(0, 500))
axes[1, 0].set_title('ELI5 - Before Length Matching')
axes[1, 0].set_xlabel('Length (words)')
axes[1, 0].legend()

# ELI5 - After matching
axes[1, 1].hist(eli5_matched[eli5_matched['label']=='human']['text_len'],
                bins=50, alpha=0.7, label='Human', range=(0, 500))
axes[1, 1].hist(eli5_matched[eli5_matched['label']=='llm']['text_len'],
                bins=50, alpha=0.7, label='LLM', range=(0, 500))
axes[1, 1].set_title('ELI5 - After Length Matching')
axes[1, 1].set_xlabel('Length (words)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("✅ STAGE 1 COMPLETE - DATA PREPARED FOR DETECTOR TRAINING")
print("="*70)
print("\n📋 Summary:")
print(f"   HC3 Train: {len(hc3_train):,} | Test: {len(hc3_test):,}")
print(f"   ELI5 Train: {len(eli5_train):,} | Test: {len(eli5_test):,}")
